# Sahil-e-Baseera: Motorcycle ROI and Helmet Detection

**Two-stage CCTV inference pipeline**

Stage 1: YOLO11s motorcycle detection. Stage 2: fine-tuned helmet detection on expanded motorcycle regions.

**Input:** CCTV video from Google Drive. **Output:** Annotated MP4 video. **Environment:** Google Colab.

The original Python logic is unchanged; only Markdown headings and cell organization have been added.


## 1. Install Dependencies


In [ ]:
# ================================================================
# SAHIL-E-BASEERA
# TWO-STAGE MOTORCYCLE ROI + HELMET DETECTION PIPELINE
#
# Stage 1: YOLO11s COCO motorcycle detection
# Stage 2: Fine-tuned helmet detector (best.pt)
#
# Input : Google Drive CCTV video
# Output: Annotated MP4 video
#
# Google Colab | SINGLE CELL
# ================================================================

# ================================================================
# 1. INSTALL DEPENDENCIES
# ================================================================

import sys
import subprocess

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "ultralytics",
    "gdown",
    "opencv-python-headless",
    "tqdm"
])



## 2. Imports


In [ ]:
# ================================================================
# 2. IMPORTS
# ================================================================

import os
import cv2
import torch
import gdown
import shutil
import numpy as np

from pathlib import Path
from tqdm.auto import tqdm
from ultralytics import YOLO
from google.colab import files
from IPython.display import Video, display



## 3. Configuration


In [ ]:
# ================================================================
# 3. CONFIGURATION
# ================================================================

# Reuse your previously downloaded video if it exists.
# Otherwise, download it from this Google Drive link.

VIDEO_URL = (
    "https://drive.google.com/file/d/1BE814oF_CxTjL3dyTH4dGb2G9IbH_TOf/view?usp=sharing"
)

INPUT_VIDEO = "/content/input_video.mp4"

OUTPUT_VIDEO = "/content/sahil_roi_helmet_output.mp4"

# Stage 1: Motorcycle detection
VEHICLE_MODEL = "yolo11s.pt"

MOTORCYCLE_CLASS = 3

MOTORCYCLE_CONF = 0.25

VEHICLE_IMGSZ = 1280

# Stage 2: Helmet detection
HELMET_CONF = 0.30

HELMET_IMGSZ = 640

# ROI expansion factors
#
# Motorcycle boxes often exclude the rider's head.
# Expand the box upward and sideways to capture the rider.

ROI_LEFT_EXPANSION = 0.30
ROI_RIGHT_EXPANSION = 0.30
ROI_TOP_EXPANSION = 1.20
ROI_BOTTOM_EXPANSION = 0.10

# Suppress duplicate helmet detections
HELMET_NMS_IOU = 0.45

# Drawing
DRAW_MOTORCYCLE_ROI = True
DRAW_HELMET_BOXES = True

# Optional: process only a short preview first.
# Set to None to process the entire video.

MAX_FRAMES = None



## 4. Gpu Check


In [ ]:
# ================================================================
# 4. GPU CHECK
# ================================================================

DEVICE = 0 if torch.cuda.is_available() else "cpu"

print("=" * 65)
print("SAHIL-E-BASEERA | TWO-STAGE HELMET DETECTION")
print("=" * 65)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: No GPU detected. Processing will be slow.")



## 5. Load Stage 1 Motorcycle Detector


In [ ]:
# ================================================================
# 5. LOAD STAGE 1 MOTORCYCLE DETECTOR
# ================================================================

print("\nLoading YOLO11s motorcycle detector...")

vehicle_model = YOLO(VEHICLE_MODEL)

print("Vehicle model loaded.")
print("Motorcycle class:", vehicle_model.names[MOTORCYCLE_CLASS])



## 6. Load Stage 2 Trained Helmet Model


In [ ]:
# ================================================================
# 6. LOAD STAGE 2 TRAINED HELMET MODEL
# ================================================================

print("\nUpload your fine-tuned helmet model (best.pt).")

uploaded = files.upload()

pt_files = [
    filename for filename in uploaded
    if filename.lower().endswith(".pt")
]

if not pt_files:
    raise ValueError(
        "No .pt model uploaded. Upload your trained best.pt."
    )

HELMET_MODEL_PATH = pt_files[0]

helmet_model = YOLO(HELMET_MODEL_PATH)

print("\nHelmet model loaded successfully.")
print("Classes:", helmet_model.names)

# Identify helmet class IDs by name rather than assuming order.

def normalize_name(name):
    return str(name).lower().replace("-", "_").replace(" ", "_")

helmet_classes = {
    int(class_id): normalize_name(name)
    for class_id, name in helmet_model.names.items()
}

required = {"with_helmet", "without_helmet"}

if set(helmet_classes.values()) != required:
    raise ValueError(
        "Expected classes with_helmet and without_helmet. "
        f"Found: {helmet_classes}"
    )



## 7. Download Or Reuse Cctv Video


In [ ]:
# ================================================================
# 7. DOWNLOAD OR REUSE CCTV VIDEO
# ================================================================

if os.path.exists(INPUT_VIDEO) and os.path.getsize(INPUT_VIDEO) > 0:

    print("\nUsing existing CCTV video:", INPUT_VIDEO)

else:

    print("\nDownloading CCTV video from Google Drive...")

    gdown.download(
        url=VIDEO_URL,
        output=INPUT_VIDEO,
        fuzzy=True,
        quiet=False
    )

    if not os.path.exists(INPUT_VIDEO):
        raise FileNotFoundError(
            "Video download failed. Check your Google Drive link."
        )



## 8. Open Video


In [ ]:
# ================================================================
# 8. OPEN VIDEO
# ================================================================

cap = cv2.VideoCapture(INPUT_VIDEO)

if not cap.isOpened():
    raise RuntimeError("Unable to open CCTV video.")

fps = cap.get(cv2.CAP_PROP_FPS)

if fps <= 0:
    fps = 25.0

width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

frames_to_process = (
    min(total_frames, MAX_FRAMES)
    if MAX_FRAMES is not None
    else total_frames
)

print("\nVIDEO INFORMATION")
print("Resolution:", width, "x", height)
print("FPS:", fps)
print("Total frames:", total_frames)
print("Processing frames:", frames_to_process)



## 9. Video Writer


In [ ]:
# ================================================================
# 9. VIDEO WRITER
# ================================================================

fourcc = cv2.VideoWriter_fourcc(*"mp4v")

writer = cv2.VideoWriter(
    OUTPUT_VIDEO,
    fourcc,
    fps,
    (width, height)
)

if not writer.isOpened():
    raise RuntimeError("Failed to create output video.")



## 10. Roi Expansion


In [ ]:
# ================================================================
# 10. ROI EXPANSION
# ================================================================

def expand_motorcycle_roi(box, frame_width, frame_height):
    """
    Expand motorcycle bounding box to include the rider.

    Input:
        box = (x1, y1, x2, y2)

    Output:
        Expanded ROI coordinates clipped to frame.
    """

    x1, y1, x2, y2 = map(float, box)

    box_width = x2 - x1
    box_height = y2 - y1

    new_x1 = x1 - ROI_LEFT_EXPANSION * box_width
    new_x2 = x2 + ROI_RIGHT_EXPANSION * box_width

    new_y1 = y1 - ROI_TOP_EXPANSION * box_height
    new_y2 = y2 + ROI_BOTTOM_EXPANSION * box_height

    new_x1 = max(0, int(new_x1))
    new_y1 = max(0, int(new_y1))

    new_x2 = min(frame_width, int(new_x2))
    new_y2 = min(frame_height, int(new_y2))

    return new_x1, new_y1, new_x2, new_y2



## 11. Global Non-Maximum Suppression


In [ ]:
# ================================================================
# 11. GLOBAL NON-MAXIMUM SUPPRESSION
# ================================================================

def calculate_iou(box_a, box_b):

    ax1, ay1, ax2, ay2 = box_a
    bx1, by1, bx2, by2 = box_b

    ix1 = max(ax1, bx1)
    iy1 = max(ay1, by1)

    ix2 = min(ax2, bx2)
    iy2 = min(ay2, by2)

    intersection = max(0, ix2 - ix1) * max(0, iy2 - iy1)

    area_a = max(0, ax2 - ax1) * max(0, ay2 - ay1)
    area_b = max(0, bx2 - bx1) * max(0, by2 - by1)

    union = area_a + area_b - intersection

    if union <= 0:
        return 0.0

    return intersection / union


def global_helmet_nms(detections, iou_threshold=0.45):
    """
    Remove duplicate helmet/head detections from overlapping ROIs.

    detections:
        [
            {
                "box": (x1,y1,x2,y2),
                "conf": float,
                "class_id": int
            }
        ]
    """

    if not detections:
        return []

    detections = sorted(
        detections,
        key=lambda d: d["conf"],
        reverse=True
    )

    kept = []

    for detection in detections:

        duplicate = False

        for previous in kept:

            overlap = calculate_iou(
                detection["box"],
                previous["box"]
            )

            # Class-agnostic NMS prevents the same head from
            # being marked both with_helmet and without_helmet.
            if overlap > iou_threshold:
                duplicate = True
                break

        if not duplicate:
            kept.append(detection)

    return kept



## 12. Drawing Helpers


In [ ]:
# ================================================================
# 12. DRAWING HELPERS
# ================================================================

COLORS = {
    "with_helmet": (0, 200, 0),
    "without_helmet": (0, 0, 255)
}

def draw_label(frame, text, x, y, color):

    font = cv2.FONT_HERSHEY_SIMPLEX
    scale = 0.6
    thickness = 2

    (text_w, text_h), baseline = cv2.getTextSize(
        text,
        font,
        scale,
        thickness
    )

    label_top = max(0, y - text_h - baseline - 10)
    label_bottom = label_top + text_h + baseline + 10

    cv2.rectangle(
        frame,
        (x, label_top),
        (x + text_w + 12, label_bottom),
        color,
        -1
    )

    cv2.putText(
        frame,
        text,
        (x + 6, label_bottom - baseline - 5),
        font,
        scale,
        (255, 255, 255),
        thickness,
        cv2.LINE_AA
    )



## 13. Main Two-Stage Inference


In [ ]:
# ================================================================
# 13. MAIN TWO-STAGE INFERENCE
# ================================================================

print("\nStarting two-stage ROI helmet detection...")

processed_frames = 0
total_motorcycle_detections = 0
total_helmet_detections = 0

progress = tqdm(
    total=frames_to_process if frames_to_process > 0 else None,
    desc="Processing CCTV"
)

try:

    while True:

        if MAX_FRAMES is not None and processed_frames >= MAX_FRAMES:
            break

        ret, frame = cap.read()

        if not ret:
            break

        # --------------------------------------------------------
        # STAGE 1: MOTORCYCLE DETECTION
        # --------------------------------------------------------

        vehicle_results = vehicle_model.predict(
            source=frame,
            classes=[MOTORCYCLE_CLASS],
            conf=MOTORCYCLE_CONF,
            imgsz=VEHICLE_IMGSZ,
            device=DEVICE,
            verbose=False
        )[0]

        motorcycle_boxes = []

        if vehicle_results.boxes is not None:

            for box in vehicle_results.boxes:

                xyxy = box.xyxy[0].cpu().numpy()

                motorcycle_boxes.append(xyxy)

        total_motorcycle_detections += len(motorcycle_boxes)

        # --------------------------------------------------------
        # STAGE 2: ROI EXTRACTION + HELMET DETECTION
        # --------------------------------------------------------

        helmet_detections = []
        expanded_rois = []

        for motorcycle_box in motorcycle_boxes:

            rx1, ry1, rx2, ry2 = expand_motorcycle_roi(
                motorcycle_box,
                width,
                height
            )

            if rx2 <= rx1 or ry2 <= ry1:
                continue

            expanded_rois.append(
                (rx1, ry1, rx2, ry2)
            )

            # Crop complete motorcycle + rider region.
            roi = frame[ry1:ry2, rx1:rx2]

            if roi.size == 0:
                continue

            helmet_results = helmet_model.predict(
                source=roi,
                conf=HELMET_CONF,
                imgsz=HELMET_IMGSZ,
                device=DEVICE,
                verbose=False
            )[0]

            if helmet_results.boxes is None:
                continue

            for helmet_box in helmet_results.boxes:

                local_box = (
                    helmet_box.xyxy[0].cpu().numpy()
                )

                hx1, hy1, hx2, hy2 = local_box

                # Convert ROI-local coordinates to
                # original full-frame coordinates.
                global_box = (
                    int(hx1 + rx1),
                    int(hy1 + ry1),
                    int(hx2 + rx1),
                    int(hy2 + ry1)
                )

                class_id = int(
                    helmet_box.cls[0].item()
                )

                confidence = float(
                    helmet_box.conf[0].item()
                )

                helmet_detections.append({
                    "box": global_box,
                    "conf": confidence,
                    "class_id": class_id
                })

        # --------------------------------------------------------
        # REMOVE DUPLICATE DETECTIONS
        # --------------------------------------------------------

        helmet_detections = global_helmet_nms(
            helmet_detections,
            iou_threshold=HELMET_NMS_IOU
        )

        total_helmet_detections += len(helmet_detections)

        # --------------------------------------------------------
        # DRAW EXPANDED MOTORCYCLE ROIS
        # --------------------------------------------------------

        if DRAW_MOTORCYCLE_ROI:

            for rx1, ry1, rx2, ry2 in expanded_rois:

                cv2.rectangle(
                    frame,
                    (rx1, ry1),
                    (rx2, ry2),
                    (255, 200, 0),
                    2
                )

        # --------------------------------------------------------
        # DRAW HELMET DETECTIONS
        # --------------------------------------------------------

        if DRAW_HELMET_BOXES:

            for detection in helmet_detections:

                x1, y1, x2, y2 = detection["box"]

                class_id = detection["class_id"]
                confidence = detection["conf"]

                class_name = helmet_classes[class_id]

                color = COLORS.get(
                    class_name,
                    (255, 255, 255)
                )

                cv2.rectangle(
                    frame,
                    (x1, y1),
                    (x2, y2),
                    color,
                    2
                )

                label = (
                    f"{class_name} {confidence:.2f}"
                )

                draw_label(
                    frame,
                    label,
                    x1,
                    y1,
                    color
                )

        # --------------------------------------------------------
        # WRITE FRAME
        # --------------------------------------------------------

        writer.write(frame)

        processed_frames += 1

        progress.update(1)

finally:

    cap.release()
    writer.release()
    progress.close()



## 14. Results


In [ ]:
# ================================================================
# 14. RESULTS
# ================================================================

print("\n" + "=" * 65)
print("PROCESSING COMPLETED")
print("=" * 65)

print("Frames processed:", processed_frames)

print(
    "Total motorcycle detections across frames:",
    total_motorcycle_detections
)

print(
    "Total helmet/head detections across frames:",
    total_helmet_detections
)

print("Output video:", OUTPUT_VIDEO)



## 15. Create Small Video Preview


In [ ]:
# ================================================================
# 15. CREATE SMALL VIDEO PREVIEW
# ================================================================

PREVIEW_VIDEO = "/content/sahil_roi_preview.mp4"

print("\nCreating 20-second preview...")

subprocess.run([
    "ffmpeg",
    "-y",
    "-loglevel", "error",
    "-i", OUTPUT_VIDEO,
    "-t", "20",
    "-vf", "scale=1280:-2",
    "-c:v", "libx264",
    "-preset", "veryfast",
    "-crf", "25",
    "-pix_fmt", "yuv420p",
    "-movflags", "+faststart",
    "-an",
    PREVIEW_VIDEO
], check=True)

display(
    Video(
        PREVIEW_VIDEO,
        embed=True,
        width=900
    )
)



## 16. Download Complete Video Locally


In [ ]:
# ================================================================
# 16. DOWNLOAD COMPLETE VIDEO LOCALLY
# ================================================================

print("\nDownloading complete annotated video...")

files.download(OUTPUT_VIDEO)

## 17. Verify Generated Video

Check the output video file and its size.


In [ ]:
import os

video_path = "/content/sahil_roi_helmet_output.mp4"

if os.path.exists(video_path):
    size = os.path.getsize(video_path) / (1024 ** 2)
    print(f"Video found! Size: {size:.2f} MB")
else:
    print("Video not found. Runtime storage may have been reset.")

## 18. Save Results to Google Drive

Mount Google Drive and copy the generated annotated video.


In [ ]:
from google.colab import drive
import shutil
import os

drive.mount("/content/drive")

source = "/content/sahil_roi_helmet_output.mp4"
destination = "/content/drive/MyDrive/sahil_roi_helmet_output.mp4"

if os.path.exists(source):
    shutil.copy2(source, destination)
    print("Video saved successfully to Google Drive!")
else:
    print("Output video not found.")